## Making initial workflows 

`trovolut` works by creating initial workflows which then can be optimized using `openevolve`. The `filemaker` module contains tools to create these initial workflows. For example, we can define code to run Trotter dynamics with a predefined product formula function in `trovolut.qm.dynamics`

In [28]:
import trovolut as trv
import warnings
warnings.filterwarnings("ignore")

In [29]:
def run_dynamics(Hs, t, psi0):
    """
    Args:
        Hs: list of Hamiltonian terms (numpy arrays)
        t: total time
        psi0: initial state (numpy array)
    """
    r = 10
    U_trotter = trv.qm.dynamics.product_formula(Hs, t, r)
    psi_final = U_trotter @ psi0
    return psi_final


We can then use the `trv.filemaker.Workflow` class to create a file with these functions. We use the `trv.filemaker.EVO_START` and `trv.filemaker.EVO_END` markers to indicate where `openevolve` should optimize the code.

In [30]:
init_program = trv.filemaker.Workflow(
    [
        trv.filemaker.EVO_START,
        run_dynamics,
        trv.filemaker.EVO_END,
    ],
    out_path="opt/init_program.py",
).build()

print("Wrote", init_program)

Wrote opt/init_program.py


In [31]:
from IPython.display import display, Markdown
import re

# Preview the generated code
display(Markdown(f"### Preview of `{init_program}`:"))
lines = re.findall(r'^(?:#.*|import .+|from .+|def .+:)', open(init_program).read(), re.M)
preview = sum(([line, "    ..."] if line.lstrip().startswith("def ") else [line] for line in lines), [])
display(Markdown("```python\n" + "\n".join(preview) + "\n```"))

### Preview of `opt/init_program.py`:

```python
import numpy as np
import scipy
import trovolut as trv
# EVOLVE-BLOCK-START
def _suzuki_step(H_terms, dt, order=1):
    ...
def product_formula(H_terms, t, r, order=1):
    ...
def product_formula(H, t, r):
    ...
def run_dynamics(Hs, t, psi0):
    ...
# EVOLVE-BLOCK-END
```

We define an evaluation function that compares the time evolution to the exact solution (and penalize if the code takes longer than one minute to run).

In [39]:
def get_hamiltonian():
    from rdkit import Chem
    from rdkit.Chem import AllChem

    mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.UFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    xyz = (
        f"{len(atoms)}\ndmds generated by RDKit\n"
        + "\n".join(
            f"{s} {conf.GetAtomPosition(i).x:.8f} "
            f"{conf.GetAtomPosition(i).y:.8f} "
            f"{conf.GetAtomPosition(i).z:.8f}"
            for i, s in enumerate(atoms)
        )
    )

    H = trv.qm.ElectronicHamiltonian.from_xyz(
        xyz_source=xyz,
        basis="sto-3g",
        charge=0,
        spin_mult=1,
        use_gpu4pyscf=False,
        freeze_core=True,
        n_core_keep=2,
        n_virt_keep=1,
    ).to_qubits()

    return [t["coef"] * t["matrix"] for t in H.terms()]

hams = get_hamiltonian()[1:]  # remove identity term
print("Hamiltonian terms:", len(hams))
print("Size of hamiltonian matrices:", hams[0].shape)

converged SCF energy = -864.654938704692
Hamiltonian terms: 75
Size of hamiltonian matrices: (64, 64)


In [ ]:
import numpy as np
import scipy
import concurrent.futures

def trotter_acc(path):
    
    # compare run_dynamics vs exact solution
    import importlib.util
    spec = importlib.util.spec_from_file_location("init_program", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    
    # parameters
    t, trials = 100, 10
    hams = get_hamiltonian()[1:]  # remove identity term

    # dynamics
    U_exact = scipy.linalg.expm(-1j * sum(hams) * t)

    dim = U_exact.shape[0]
    psi0_batch = np.random.rand(trials, dim) + 1j * np.random.rand(trials, dim)
    psi0_batch = psi0_batch / np.linalg.norm(psi0_batch, axis=1)[:, np.newaxis]

    psi_exact_batch = psi0_batch @ U_exact.T

    psi_trotter_batch = np.array([
        module.run_dynamics(hams, t, psi0_batch[i])
        for i in range(trials)
    ])

    # compute errors
    err = np.linalg.norm(psi_trotter_batch - psi_exact_batch, axis=1) 
    errs = err.tolist()

    # compute accuracy and its std
    mean_err = np.mean(errs)
    acc = 1.0 / (1.0 + mean_err)

    var_err = np.var(errs) / trials
    std_acc = np.sqrt(var_err) / (1.0 + mean_err)**2

    return {"combined_score": acc, "std_score": std_acc}



def run_with_timeout(func, args=(), kwargs=None, timeout_seconds=30):
    if kwargs is None:
        kwargs = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(func, *args, **kwargs)
        try:
            return future.result(timeout=timeout_seconds)
        except concurrent.futures.TimeoutError:
            raise TimeoutError(f"Function timed out after {timeout_seconds} seconds")


def evaluate(program_path):
    """
    OpenEvolve evaluator for Trotter accuracy with a hard timeout.
    If evaluation takes longer than TIMEOUT_SECONDS, return a big penalty score.
    """
    try:
        result = run_with_timeout(
            trotter_acc,  
            args=(program_path,),
            timeout_seconds=30,
        )

        combined = float(result.get("combined_score", -100.0))
        std = float(result.get("std_score", 0.0))

        return {
            "combined_score": combined,
            "std_score": std,
        }

    except Exception as e:
        print(f"Evaluation crashed: {e}")
        return {
            "combined_score": -100.0,
            "std_score": 0.0,
        }

acc = evaluate(init_program)
print(f"Trotter accuracy: {acc['combined_score']} ± {acc['std_score']}")

converged SCF energy = -864.654938691007
Evaluation crashed: name 'mats' is not defined
Trotter accuracy: -100.0 ± 0.0


We create another file with such an evaluation function.

In [ ]:
def main_eval():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--program_path", type=str, required=True)
    args = parser.parse_args()
    result = evaluate(args.program_path)
    print(f"combined_score: {result['combined_score']}")
    print(f"std_score: {result['std_score']}")


evaluator_path = trv.filemaker.Workflow(
    [
        get_hamiltonian,
        trotter_acc,
        run_with_timeout,
        evaluate
    ],
    out_path="opt/evaluator.py",
    imports=[
        "import concurrent.futures",
        "import numpy as np",
        "import scipy",
        "import trovolut as trv"
    ],
    main_func=main_eval,
).build()

In [8]:
import subprocess

result = subprocess.run(
    ["uv", "run", "python", str(evaluator_path), "--program_path", str(init_program)],
    capture_output=True,
    text=True
)

print(result.stdout)

converged SCF energy = -864.654938672564
combined_score: 0.624614864453918
std_score: 0.008803894734333967



Now, feeding it into `openevolve`, we can optimize the workflow.

In [9]:
import nest_asyncio
nest_asyncio.apply()

from openevolve import run_evolution
from openevolve.config import Config, LLMModelConfig, PromptConfig

config = Config()
prompt = PromptConfig(system_message="You are an expert programmer specializing in trotterization for quantum evolution. Your task is to maximize the Trotter simulation accuracy of the provided code, but the implementation must run in less than one minute.")

config.prompt = prompt
config.llm.models = [LLMModelConfig(name="gpt-4o")]
config.evaluator.cascade_evaluation = False

# Optimize Trotter circuit using OpenEvolve
result = run_evolution(
    initial_program=init_program,
    evaluator=evaluator_path,
    iterations=10,
    config=config
)


2025-11-10 02:52:44,840 - INFO - Logging to /var/folders/x9/8zgqvcbx66q2mjvw0wjdx_fw0000gn/T/openevolve_li1j3dow/logs/openevolve_20251110_025244.log
2025-11-10 02:52:44,841 - INFO - Set random seed to 42 for reproducibility
2025-11-10 02:52:44,893 - INFO - Initialized OpenAI LLM with model: gpt-4o
2025-11-10 02:52:44,893 - INFO - Initialized LLM ensemble with models: gpt-4o (weight: 1.00)
2025-11-10 02:52:44,897 - INFO - Initialized prompt sampler
2025-11-10 02:52:44,899 - INFO - Set custom templates: system=evaluator_system_message, user=None
2025-11-10 02:52:44,899 - INFO - Initialized program database with 0 programs
2025-11-10 02:52:44,903 - INFO - Successfully loaded evaluation function from opt/evaluator.py
2025-11-10 02:52:44,903 - INFO - Initialized evaluator with opt/evaluator.py
2025-11-10 02:52:44,903 - INFO - Initialized OpenEvolve with opt/init_program.py
2025-11-10 02:52:44,904 - INFO - Adding initial program to database


converged SCF energy = -864.654938704692


2025-11-10 02:52:46,830 - INFO - Evaluated program 39ea0c64-993d-42da-8569-d2040b941c23 in 1.92s: combined_score=0.6119, std_score=0.0062
2025-11-10 02:52:46,830 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 5, 'diversity': 0}
2025-11-10 02:52:46,848 - INFO - Initialized process parallel controller with 1 workers
2025-11-10 02:52:46,851 - INFO - Started process pool with 1 processes
2025-11-10 02:52:46,851 - INFO - Using island-based evolution with 5 islands
2025-11-10 02:52:46,852 - INFO - Island Status:
2025-11-10 02:52:46,852 - INFO -  * Island 0: 1 programs, best=0.6119, avg=0.6119, diversity=0.00, gen=0 (best: 39ea0c64-993d-42da-8569-d2040b941c23)
2025-11-10 02:52:46,852 - INFO -    Island 1: 0 programs, best=0.0000, avg=0.0000, diversity=0.00, gen=0
2025-11-10 02:52:46,853 - INFO -    Island 2: 0 programs, best=0.0000, avg=0.0000, diversity=0.00, gen=0
2025-11-10 02:52:46,853 - INFO -    Island 3: 0 programs, best=0.0000, avg=0.0000, diversity=0.00, gen=0
2025

converged SCF energy = -864.654938672564


/Users/luismantilla/Desktop/UofT/NVIDIA/trovolut/.venv/lib/python3.13/site-packages/pyscf/lib/misc.py:1238: Warning: 
        Seeding the random number generator with a non-constant value
        inside Legate can lead to undefined behavior and/or errors when
        the program is executed with multiple ranks.
  random.seed()
<cupynumeric internals>:284: RuntimeWarning: cuPyNumeric has not implemented numpy.kron and is falling back to canonical NumPy. You may notice significantly decreased performance for this function call.

Set CUPYNUMERIC_FALLBACK_STACKTRACE=1 and re-run to include a full stack trace with this warning.
2025-11-10 02:53:01,370 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 5}
2025-11-10 02:53:01,372 - INFO - New best program 8a17394a-15fa-4a06-ade7-ae4b684cda33 replaces 39ea0c64-993d-42da-8569-d2040b941c23 (combined_score: 0.6119 → 0.6253, +0.0135)
2025-11-10 02:53:01,372 - INFO - Iteration 1: Program 8a17394a-15fa-4a06-ade7-ae4b68

converged SCF energy = -864.654938666712


... (11 outputs left out)

converged SCF energy = -864.654938665794


2025-11-10 02:54:41,589 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 7, 'diversity': 4}
2025-11-10 02:54:41,591 - INFO - Iteration 9: Program 6a269d8a-afa3-49b5-b6a0-8eb87677720c (parent: 4c1243f0-ecc5-48a4-b840-b511bbf55354) completed in 13.09s
2025-11-10 02:54:41,591 - INFO - Metrics: combined_score=0.6076, std_score=0.0051


converged SCF energy = -864.654938672795


2025-11-10 02:55:09,825 - INFO - Iteration 10: Program 2201d999-41cd-4603-9e7c-56fd6f8c3577 (parent: 84b11a51-0e8c-42c9-89c5-028ff4b10d26) completed in 28.25s
2025-11-10 02:55:09,827 - INFO - Metrics: combined_score=0.4322, std_score=0.0018
2025-11-10 02:55:09,828 - INFO - ✅ Evolution completed - Maximum iterations reached
2025-11-10 02:55:10,547 - INFO - Stopped process pool
2025-11-10 02:55:10,548 - INFO - Using tracked best program: 84b11a51-0e8c-42c9-89c5-028ff4b10d26
2025-11-10 02:55:10,548 - INFO - Evolution complete. Best program has metrics: combined_score=0.8755, std_score=0.0027
2025-11-10 02:55:10,550 - INFO - Saved best program to /var/folders/x9/8zgqvcbx66q2mjvw0wjdx_fw0000gn/T/openevolve_li1j3dow/best/best_program.py with program info to /var/folders/x9/8zgqvcbx66q2mjvw0wjdx_fw0000gn/T/openevolve_li1j3dow/best/best_program_info.json


After mutating the code for 10 cycles, we get the following optimized trotterization code:

In [13]:
display(Markdown(f"```python\n{result.best_code}\n```"))

```python
import numpy as np
import scipy
import trovolut as trv

# EVOLVE-BLOCK-START

def product_formula(H, t, r):
    """Second-order Trotter-Suzuki decomposition."""
    dt = t / r
    U_trotter = np.eye(H[0].shape[0], dtype=complex)

    for _ in range(r):
        for H_j in H:
            U_j_half = scipy.linalg.expm(-0.5j * H_j * dt)
            U_trotter = U_j_half @ U_trotter @ U_j_half

    return U_trotter

def run_dynamics(mats, t, psi0):
    """
    Args:
        mats: list of Hamiltonian terms (numpy arrays)
        t: total time
        psi0: initial state (numpy array)
    """
    r = 20  # Increase r for better accuracy
    U_trotter = product_formula(mats, t, r)
    psi_final = U_trotter @ psi0
    return psi_final

# EVOLVE-BLOCK-END


```

In [ ]:
# save best program
with open("opt/best_trotter_program.py", "w") as f:
    f.write(result.best_code)